Entity	            Type	
income	            Numeric	
income_frequency	Category
loan_amount	        Numeric	
loan_type	        Category	
tenure	            Numeric	
interest_rate	    Numeric
credit_score	    Numeric	
dependents	        Numeric	
existing_loan	    Numeric

In [731]:
import re
import pandas as pd

In [732]:
entity_schema = {
    "income":{
        "type":"numeric",
        "discription":"Applicant's income",
        "unit":["ruppes","thousand","lakh"],
        "required":True
    },
    
    "income_frequency":{
        "type":"category",
        "discription":"Applicant's income frequency",
        "allowed_values":["monthly","yearly"],
        "required":False
    },
    "loan_amount":{
        "type":"numeric",
        "discription":"Requsted loan amount",
        "unit":["ruppes","thousand","lakh"],
        "required":True
    },
    "loan_type":{
        "type":"category",
        "discription":"Type of loan",
        "allowed_values":[
            "Personal_loan",
            "business_loan",
            "car_loan",
            "home_loan",
            "education_loan"
        ],
        "required":False
    },
    "tenure":{
        "type":"numeric",
        "discription":"Loan repayment period",
        "unit":["years","months"],
        "required":True
    },
    "interest_rate": {
        "type": "numeric",
        "description": "Loan interest rate",
        "unit": ["percent"],
        "required": False
    },

    "credit_score": {
        "type": "numeric",
        "description": "Applicant's credit/CIBIL score",
        "unit": None,
        "required": False
    },

    "dependents": {
        "type": "numeric",
        "description": "Number of dependents",
        "unit": None,
        "required": False
    },

    "existing_loan": {
        "type": "numeric",
        "description": "Existing loan or EMI amount",
        "unit": ["rupees", "thousand", "lakh"],
        "required": False
    }
}
        
        
print(entity_schema)



{'income': {'type': 'numeric', 'discription': "Applicant's income", 'unit': ['ruppes', 'thousand', 'lakh'], 'required': True}, 'income_frequency': {'type': 'category', 'discription': "Applicant's income frequency", 'allowed_values': ['monthly', 'yearly'], 'required': False}, 'loan_amount': {'type': 'numeric', 'discription': 'Requsted loan amount', 'unit': ['ruppes', 'thousand', 'lakh'], 'required': True}, 'loan_type': {'type': 'category', 'discription': 'Type of loan', 'allowed_values': ['Personal_loan', 'business_loan', 'car_loan', 'home_loan', 'education_loan'], 'required': False}, 'tenure': {'type': 'numeric', 'discription': 'Loan repayment period', 'unit': ['years', 'months'], 'required': True}, 'interest_rate': {'type': 'numeric', 'description': 'Loan interest rate', 'unit': ['percent'], 'required': False}, 'credit_score': {'type': 'numeric', 'description': "Applicant's credit/CIBIL score", 'unit': None, 'required': False}, 'dependents': {'type': 'numeric', 'description': 'Number 

In [733]:
print(entity_schema.keys())

dict_keys(['income', 'income_frequency', 'loan_amount', 'loan_type', 'tenure', 'interest_rate', 'credit_score', 'dependents', 'existing_loan'])


In [734]:
applicant_data = {
    "income":None,
    "income_frequency":None,
    "loan_amount":None,
    "loan_type":None,
    "tenure":None,
    "interest_rate":None,
    "credit_score":None,
    "dependents":None,
    "existing_loan":None
}

In [735]:
user_text = """
I earn ₹60,000 per month and I want a ₹10 lakh personal loan
for 5 years. My CIBIL score is 750 and I have 2 dependents.
"""

In [736]:
print(user_text)


I earn ₹60,000 per month and I want a ₹10 lakh personal loan
for 5 years. My CIBIL score is 750 and I have 2 dependents.



In [737]:
# result = extract_existing_loan(user_text)
# print(result)

In [738]:
import re

def find_income_expression(text):

    text = text.lower().strip()

    # Income-related phrases
    income_patterns = [
        r"\b(?:my\s+)?(?:monthly\s+)?income\s+(?:is|of|around|about)?\s*",
        r"\b(?:my\s+)?(?:monthly\s+)?salary\s+(?:is|of|around|about)?\s*",
        r"\bi\s+(?:earn|make|receive|get|take home)\s*",
        r"\bi\s+am\s+earning\s*",
        r"\bearning\s+(?:an?\s+)?income\s+of\s*"
    ]

    # Money expression
    money_pattern = r"""
        (?P<currency>₹|rs\.?|inr|\$|usd)?
        \s*
        (?P<number>\d+(?:,\d+)*(?:\.\d+)?)
        \s*
        (?P<unit>lakhs?|lacs?|crores?|k|thousand|million)?
    """

    for income_pattern in income_patterns:

        for phrase_match in re.finditer(income_pattern, text):

            # Search for a money value immediately after
            # the income-related phrase.
            remaining_text = text[phrase_match.end():]

            money_match = re.match(
                r"\s*" + money_pattern,
                remaining_text,
                re.VERBOSE
            )

            if money_match:

                return {
                    "currency": money_match.group("currency"),
                    "number": money_match.group("number"),
                    "unit": money_match.group("unit")
                }

    return None

In [739]:
text="I earn ₹60,000 thousand per month"

find_income_expression(text)

{'currency': '₹', 'number': '60,000', 'unit': 'thousand'}

In [740]:
def normalize_money_value(result):
    
    if result is None:
        return None
    
    number = float(result['number'].replace(",",""))
    unit = result['unit']
    
    multipliers = {
        "k":1_000,
        "thousand":1_000,
        "lakh":1_00_000,
        "lakhs":1_00_000,
        "lac":1_00_000,
        "lacs":1_00_000,
        "crore":1_00_00_000,
        "crores":1_00_00_000,
        "million":10_00_000
    }
    
    if unit:
        number = number * multipliers.get(unit,1)
    return number
    

In [741]:
test_data={'currancy': '₹', 'number': '60', 'unit': 'crores'}
    # {'currancy': '₹', 'number': '60,000', 'unit':None},
    # {'currancy': '₹', 'number': '60', 'unit': 'thousand'},
    # {'currancy': '₹', 'number': '60', 'unit': 'lac'},
    # {'currancy': '₹', 'number': '6.2', 'unit': 'lacs'}
    # {'currancy': '₹', 'number': '60', 'unit': 'crore'},
    # {'currancy': '₹', 'number': '60', 'unit': 'crores'},
    # {'currancy': '₹', 'number': '60', 'unit': 'million'}
    
normalize_money_value(test_data)

600000000.0

In [742]:
frequency_patterns={
    "monthly":[
        'per month',
        'monthly',
        'a month',
        'every month',
        'each month'
    ],
    "yearly":[
        'per year',
        'yearly',
        'annual',
        'annually',
        'every year',
        'each year',
        # 'a year',
        
    ],
    "weekly":[
        'per week',
        'weekly',
        'a week',
        'each week',
        'every week'
    ],
    "daily":[
        'per day',
        'each day',
        'every day',
        'daily'
    ]
        
}

print(frequency_patterns)
print(frequency_patterns.keys())

{'monthly': ['per month', 'monthly', 'a month', 'every month', 'each month'], 'yearly': ['per year', 'yearly', 'annual', 'annually', 'every year', 'each year'], 'weekly': ['per week', 'weekly', 'a week', 'each week', 'every week'], 'daily': ['per day', 'each day', 'every day', 'daily']}
dict_keys(['monthly', 'yearly', 'weekly', 'daily'])


In [743]:
# def detect_income_frequency(text):
    
#     #Normlize the text
#     text = text.lower().strip()
    
#     #itreate through  frequency patterns
#     for frequency,patterns in frequency_patterns.items():
#         for pattern in patterns:
#             if pattern in text:
#                 return frequency
#     return None
def detect_income_frequency(text):

    # Step 1: Validate input
    if not isinstance(text, str) or not text.strip():
        return None

    # Step 2: Normalize text
    text = re.sub(r"\s+", " ", text.lower().strip())

    # Step 3: Check frequency patterns
    for frequency, patterns in frequency_patterns.items():
        for pattern in patterns:
            if pattern in text:
                return frequency

    # Step 4: No frequency found
    return None  


In [744]:
text = 'I earn ₹60,000 per day'

detect_income_frequency(text)

'daily'

In [745]:
def extract_income(text):
    result = find_income_expression(text)
    income = normalize_money_value(result)
    frequency = detect_income_frequency(text)
    
    return {
        "income":income,
        "frequency":frequency
    }

In [746]:
test_sentences = [
    "I earn ₹60,000 per month",
    "My monthly salary is 60k",
    "I earn 60 thousand per month",
    "I earn 7.2 lakh per year",
    "My annual income is ₹8 lakh",
    "I earn ₹1,20,000 monthly",
    "I earn ₹80K",
    "I earn ₹10 lakh yearly",
    "I earn ₹60,000"
]

for text in test_sentences:

    result = extract_income(text)

    print(text)
    print(result)
    print("-" * 50)

I earn ₹60,000 per month
{'income': 60000.0, 'frequency': 'monthly'}
--------------------------------------------------
My monthly salary is 60k
{'income': 60000.0, 'frequency': 'monthly'}
--------------------------------------------------
I earn 60 thousand per month
{'income': 60000.0, 'frequency': 'monthly'}
--------------------------------------------------
I earn 7.2 lakh per year
{'income': 720000.0, 'frequency': 'yearly'}
--------------------------------------------------
My annual income is ₹8 lakh
{'income': 800000.0, 'frequency': 'yearly'}
--------------------------------------------------
I earn ₹1,20,000 monthly
{'income': 120000.0, 'frequency': 'monthly'}
--------------------------------------------------
I earn ₹80K
{'income': 80000.0, 'frequency': None}
--------------------------------------------------
I earn ₹10 lakh yearly
{'income': 1000000.0, 'frequency': 'yearly'}
--------------------------------------------------
I earn ₹60,000
{'income': 60000.0, 'frequency': Non

In [747]:
# import re

# def find_loan_amount_expression(text):

#     if not isinstance(text, str) or not text.strip():
#         return None

#     text = text.lower().strip()

#     money_pattern = r"""
#         (?P<currency>₹|rs\.?|inr|\$|usd)?
#         \s*
#         (?P<number>\d+(?:,\d+)*(?:\.\d+)?)
#         \s*
#         (?P<unit>lakhs?|lacs?|crores?|k|thousand|million)?
#     """

#     loan_amount_patterns = [
#         r"\bloan amount\s*(?:is|about|around|of)?\s*",
#         r"\bloan\s+of\s*",
#         r"\bborrow\s*",
#         r"\b(?:need|want|require)\s+(?:a\s+)?loan\s+(?:of\s*)?",
#         r"\b(?:need|want|require)\s+(?:to\s+)?borrow\s*",
#         r"\b(?:finance|financing)\b[^.!?]{0,50}?\bfor\s*",
#         r"\b(?:car|vehicle|house|home|property|flat)\s+worth\s*"

#         # NEW: "need ₹2 lakh for personal expenses"
#         r"\b(?:need|want|require)\s+"
#     ]
    
#     for phrase in loan_amount_patterns:
#         for phrase_match in re.finditer(phrase, text):

#             remaining_text = text[phrase_match.end():]

#             money_match = re.match(
#                 r"\s*" + money_pattern,
#                 remaining_text,
#                 re.VERBOSE
#             )

#             if money_match:
#                 return {
#                     "currency": money_match.group("currency"),
#                     "number": money_match.group("number"),
#                     "unit": money_match.group("unit")
#                 }

#     amount_before_loan_pattern = money_pattern + r"\s+loan\b"

#     for match in re.finditer(
#         amount_before_loan_pattern,
#         text,
#         re.VERBOSE
#     ):
#         prefix = text[:match.start()].strip()

#         request_pattern = (
#             r"\b(?:need|want|require|get|take|"
#             r"apply for|looking for)\b"
#         )

#         if re.search(request_pattern, prefix[-60:]):
#             return {
#                 "currency": match.group("currency"),
#                 "number": match.group("number"),
#                 "unit": match.group("unit")
#             }

#     return None

def find_loan_amount_expression(text):

    if not isinstance(text, str) or not text.strip():
        return None

    text = re.sub(r"\s+", " ", text.lower().strip())

    # Match a currency amount and its optional unit
    money_pattern = re.compile(
        r"(?P<currency>₹|rs\.?|inr|\$|usd)?\s*"
        r"(?P<number>\d+(?:,\d{3})*(?:\.\d+)?)\s*"
        r"(?P<unit>lakhs?|lacs?|crores?|k|thousand|millions?)?",
        re.IGNORECASE
    )

    # Specific contexts first, then general request phrases
    request_patterns = [
        r"\bloan amount\s*(?:is|about|around|of)?\s*",
        r"\b(?:student|education|personal|business|car|vehicle|home|house|auto)\s+loans?\s+of\s*",
        r"\bloans?\s+of\s*",
        r"\b(?:need|want|require)\s+(?:a\s+)?loans?\s+(?:of\s*)?",
        r"\b(?:need|want|require)\s+(?:to\s+)?borrow\s*",
        r"\b(?:need|want|require)\s+",
        r"\b(?:car|vehicle|house|home|property|flat)\s+worth\s*",
        r"\b(?:finance|financing)\b[^.!?]{0,60}?\bfor\s*",
    ]

    # Search for an amount immediately after a request phrase
    for pattern in request_patterns:
        for phrase_match in re.finditer(pattern, text):

            remaining_text = text[phrase_match.end():]
            money_match = money_pattern.match(remaining_text)

            if money_match:
                return {
                    "currency": money_match.group("currency"),
                    "number": money_match.group("number"),
                    "unit": money_match.group("unit")
                }

    # Fallback: currency amount following a loan-related cue.
    # Avoid amounts whose nearest relevant context is salary/income.
    money_with_currency = re.compile(
        r"(?P<currency>₹|rs\.?|inr|\$|usd)\s*"
        r"(?P<number>\d+(?:,\d{3})*(?:\.\d+)?)\s*"
        r"(?P<unit>lakhs?|lacs?|crores?|k|thousand|millions?)?",
        re.IGNORECASE
    )

    request_cue = re.compile(
        r"\b(?:need|want|require|borrow|loan|finance|financing)\b",
        re.IGNORECASE
    )

    for money_match in money_with_currency.finditer(text):

        prefix = text[:money_match.start()]
        cues = list(request_cue.finditer(prefix))

        if not cues:
            continue

        # Check the text between the latest request cue and this amount.
        context = prefix[cues[-1].end():]

        if re.search(
            r"\b(?:salary|income|earn|earning)\b",
            context
        ):
            continue

        return {
            "currency": money_match.group("currency"),
            "number": money_match.group("number"),
            "unit": money_match.group("unit")
        }

    return None


In [748]:
def extract_loan_amount(text):
    money_data = find_loan_amount_expression(text)
    loan_amount = normalize_money_value(money_data)
    
    return loan_amount

In [749]:

text = "I receive ₹35,000 per month and need ₹2 lakh for personal expenses over 2 years. My CIBIL score is 690 and I have no dependents."

print("Extracted data:", find_loan_amount_expression(text))
print("Normalized amount:", extract_loan_amount(text))

Extracted data: {'currency': '₹', 'number': '2', 'unit': 'lakh'}
Normalized amount: 200000.0


In [750]:
test_sentences = [
    "I want a loan of ₹5 lakh",
    "I need ₹10,00,000 as a loan",
    "My loan amount is 7.5 lakhs",
    "I want to borrow 2 lakh",
    "I need a loan of 60k",
    "I want a personal loan for 5 years",
    "Can I get a loan?",
    "I earn ₹60,000 per month and want a ₹5 lakh loan for 5 years"
]

for text in test_sentences:
    print("Text:", text)
    print("Loan amount:", extract_loan_amount(text))
    print("-" * 50)

Text: I want a loan of ₹5 lakh
Loan amount: 500000.0
--------------------------------------------------
Text: I need ₹10,00,000 as a loan
Loan amount: 10.0
--------------------------------------------------
Text: My loan amount is 7.5 lakhs
Loan amount: 750000.0
--------------------------------------------------
Text: I want to borrow 2 lakh
Loan amount: 200000.0
--------------------------------------------------
Text: I need a loan of 60k
Loan amount: 60000.0
--------------------------------------------------
Text: I want a personal loan for 5 years
Loan amount: None
--------------------------------------------------
Text: Can I get a loan?
Loan amount: None
--------------------------------------------------
Text: I earn ₹60,000 per month and want a ₹5 lakh loan for 5 years
Loan amount: 500000.0
--------------------------------------------------


In [751]:
text = "I earn ₹60,000 per month and want a ₹5 lakh loan for 5 years"

print(extract_loan_amount(text))

500000.0


In [752]:
loan_patterns = {
    "personal_loan": [
        "want a personal loan"
        "need a personal loan",
        "apply for a personal loan",
        "want to get a personal loan",
        "looking for a personal loan",
        "I want personal loan",
        "I need money for personal expenses",
        "can I get a personal loan",
        "I require a personal loan",
        "I want to borrow money",
        "how to apply for a personal loan",
        "I need funds for an emergency",
        "personal loan for medical expenses",
        "I want a loan for my wedding",
        "tell me about personal loans",
        "what are the personal loan options",
        "I need an unsecured loan",
        "I want to take a personal loan",
        "help me get a personal loan",
        "am I eligible for a personal loan",
        "what documents are needed for a personal loan",
        "want a personal loan",
        "need a personal loan",
        "personal loan",
        "want personal loan",
        "need personal loan",
        "for personal expenses",
    ],
    "business_loan":[
        "need a business loan",
        "apply for a business loan",
        "want to get a business loan",
        "looking for a business loan",
        "I want a business loan",
        "I need money for business expenses",
        "can I get a business loan",
        "I require a business loan",
        "how to apply for a business loan",
        "I want a business loan for my company",
        "tell me about business loan",
        "I want to take a business loan",
        "help me get a business loan",
        "am I eligible for a business loan",
        "what documents are needed for a business loan",
        "loan for my business",
        "loan for my company",
        "I need funds to start a business",
        "I need capital for my business",
        "loan for business expansion",
        "funding for my startup",
        "I want to expand my business",
        "I need working capital",
        "loan to purchase business equipment",
        "loan to open a new shop",
        "small business loan",
        "startup business loan",
        "business loan interest rate",
        "business loan eligibility",
        "business loan repayment options",
        "how can I finance my business",
        "I need money to grow my company",
        "loan for business development",
        "I want funding for my new venture",
        "what is the minimum business loan amount"
    ],
    
    "car_loan": [
        "need a car loan",
        "apply for a car loan",
        "I want to get a car loan",
        "looking for a car loan",
        "can I get a car loan",
        "I want to buy a new car",
        "I need a loan to purchase a car",
        "loan for buying a used car",
        "I want to finance my car",
        "I need car financing",
        "what are the car loan interest rates",
        "am I eligible for a car loan",
        "what documents are needed for a car loan",
        "I want to take a car loan",
        "help me get a car loan",
        "car loan repayment options",
        "I need a vehicle loan for my car",
        "how can I apply for car finance",
        "I want to buy a second hand car",
        "tell me about car loans"
    ],
        
    "home_loan": [
        "need a home loan",
        "apply for a home loan",
        "I want to get a home loan",
        "looking for a home loan",
        "can I get a home loan",
        "I want to buy a house",
        "I need a loan to purchase a house",
        "loan for buying a flat",
        "I want to purchase a new apartment",
        "I need financing for a property",
        "what are the home loan interest rates",
        "am I eligible for a home loan",
        "what documents are needed for a home loan",
        "I want to take a home loan",
        "help me get a home loan",
        "home loan repayment options",
        "I need a mortgage loan",
        "loan for construction of a house",
        "I want a loan to renovate my home",
        "tell me about home loans",
        "want home loan",
        "need home loan",
        "home loan",
        "need a loan to buy a house",
        "need money to buy a house",
        "need funds to buy a house",
        "need a loan to purchase a house",
        "need ₹25 lakh to buy a house",
        "loan to buy a house",
        "loan to purchase a house",
        "want to buy a house with a loan",
        "need financing to buy a house",
        "loan to construct my house",
        "loan to construct a house",
        "need a loan to construct my house",
        "want a loan to construct my house",
        "finance to construct a house",
        "loan for house construction",
        "home construction loan",
        
    ],
    
    "education_loan": [
        "need an education loan",
        "apply for an education loan",
        "I want to get an education loan",
        "looking for an education loan",
        "can I get an education loan",
        "I need a loan for my college fees",
        "loan for higher studies",
        "I want to study abroad and need a loan",
        "I need financial assistance for my studies",
        "loan for university tuition fees",
        "what are the education loan interest rates",
        "am I eligible for an education loan",
        "what documents are needed for an education loan",
        "I want to take an education loan",
        "help me get an education loan",
        "education loan repayment options",
        "I need funding for my master's degree",
        "loan for engineering studies",
        "loan for medical college admission",
        "tell me about student loans"
    ]
    
}

In [753]:
negation_patterns = {
    "negation" : [
        "don't want",
        "do not want",
        "don't need",
        "do not need",
        "not interested in",
        "no need for"
    ]
}

In [754]:

# # import re

# # def extract_loan_type(text):

# #     if not isinstance(text, str) or not text.strip():
# #         return None

# #     text = text.lower().strip()
# #     detected_loan_types = []

# #     for loan_type, patterns in loan_patterns.items():

# #         for phrase in patterns:

# #             phrase = phrase.lower()

# #             for match in re.finditer(re.escape(phrase), text):

# #                 position = match.start()
# #                 preceding_text = text[:position].strip()

# #                 is_negated = any(
# #                     re.search(
# #                         r"\b" + re.escape(negation) + r"\b[^.!?]*$",
# #                         preceding_text
# #                     )
# #                     for negation in negation_patterns
# #                 )

# #                 if is_negated:
# #                     continue

# #                 if loan_type not in detected_loan_types:
# #                     detected_loan_types.append(loan_type)

# #                 break

# #     return detected_loan_types if detected_loan_types else None

# import re

# def extract_loan_type(text):

#     if not isinstance(text, str) or not text.strip():
#         return None

#     text = text.lower().strip()

#     request_patterns = {
#         "personal_loan": [
#             r"\bpersonal loan\b",
#             r"\bpersonal expenses\b",
#             r"\bunsecured loan\b"
#         ],
#         "business_loan": [
#             r"\bbusiness loan\b",
#             r"\bworking capital\b",
#             r"\bbusiness expansion\b"
#         ],
#         "car_loan": [
#             r"\bcar loan\b",
#             r"\bvehicle loan\b",
#             r"\bauto loan\b"
#         ],
#         "home_loan": [
#             r"\bhome loan\b",
#             r"\bhouse loan\b",
#             r"\bmortgage loan\b",
#             r"\bconstruct(?:ion)? of (?:my |a |the )?house\b",
#             r"\bconstruct my house\b",
#             r"\bbuy a house\b",
#             r"\bpurchase a house\b"
#         ],
#         "education_loan": [
#             r"\beducation loan\b",
#             r"\bstudent loan\b",
#             r"\bloan for (?:my )?studies\b"
#         ]
#     }

#     # Prefer a loan type mentioned near an explicit request.
#     request_pattern = (
#         r"\b(?:want|need|require|apply for|get|take|"
#         r"looking for)\b"
#     )

#     for request in re.finditer(request_pattern, text):
#         following_text = text[request.start():request.start() + 100]

#         for loan_type, patterns in request_patterns.items():
#             if any(re.search(p, following_text) for p in patterns):
#                 return loan_type

#     # Fallback: recognize a loan type anywhere in the message.
#     for loan_type, patterns in request_patterns.items():
#         if any(re.search(p, text) for p in patterns):
#             return loan_type

#     return None

# def extract_loan_type(text):

#     if not isinstance(text, str) or not text.strip():
#         return None

#     text = text.lower().strip()
#     text = re.sub(r"\s+", " ", text)

#     loan_patterns = {
#         "personal_loan": [
#             r"\bpersonal loans?\b",
#             r"\bunsecured loans?\b",
#             r"\bpersonal expenses\b",
#             r"\bmoney for (?:a )?personal purpose\b"
#         ],
#         "business_loan": [
#             r"\bbusiness loans?\b",
#             r"\bworking capital\b",
#             r"\bcapital for my business\b",
#             r"\bfunds? for (?:my )?business\b",
#             r"\bbusiness expansion\b",
#             r"\bstart a business\b"
#         ],
#         "car_loan": [
#             r"\bcar loans?\b",
#             r"\bvehicle loans?\b",
#             r"\bauto loans?\b",
#             r"\bbuy (?:a |an |my )?(?:new |used |second.hand )?car\b",
#             r"\bpurchase a car\b",
#             r"\bcar financing\b"
#         ],
#         "home_loan": [
#             r"\bhome loans?\b",
#             r"\bhouse loans?\b",
#             r"\bmortgage loans?\b",
#             r"\b(?:buy|purchase|construct|build|renovate)\b"
#             r".{0,50}\b(?:house|home|flat|property)\b",
#             r"\bloan for (?:my )?(?:house|home|property)\b"
#         ],
#         "education_loan": [
#             r"\beducation loans?\b",
#             r"\bstudent loans?\b",
#             r"\bloan for (?:my )?(?:college fees|studies|tuition fees)\b",
#             r"\b(?:funding|financial assistance) for my studies\b"
#         ]
#     }

#     # Divide text into clauses so negation in one clause
#     # does not automatically reject a loan in another clause.
#     clauses = re.split(r"\bbut\b|[.!?;]", text)

#     # Common explicit negation phrases
#     negation_pattern = re.compile(
#         r"\b(?:don't|do not|doesn't|does not|"
#         r"didn't|did not|not|never|no longer)\b"
#         r".{0,40}$"
#     )

#     positive_types = []
#     negative_types = []

#     for clause in clauses:
#         clause = clause.strip()

#         if not clause:
#             continue

#         for loan_type, patterns in loan_patterns.items():
#             for pattern in patterns:
#                 for match in re.finditer(pattern, clause):

#                     phrase_start = match.start()
#                     prefix = clause[:phrase_start].strip()

#                     # Check for negation before this phrase.
#                     negated = bool(
#                         negation_pattern.search(prefix)
#                     )

#                     # "not interested in a personal loan"
#                     if re.search(
#                         r"\bnot interested in\s*$",
#                         prefix
#                     ):
#                         negated = True

#                     target = (
#                         negative_types if negated
#                         else positive_types
#                     )

#                     if loan_type not in target:
#                         target.append(loan_type)

#                     break

#     # Positive requests take priority over rejected loan types.
#     if positive_types:
#         return positive_types

#     if negative_types:
#         return None

#     return None


# import re

# def extract_requested_loan_type(text):

#     if not isinstance(text, str) or not text.strip():
#         return None

#     text = text.lower().strip()
#     text = re.sub(r"\s+", " ", text)

#     loan_patterns = {
#         "personal_loan": [
#             r"\bpersonal loans?\b",
#             r"\bunsecured loans?\b",
#             r"\bpersonal expenses\b"
#         ],
#         "business_loan": [
#             r"\bbusiness loans?\b",
#             r"\bworking capital\b",
#             r"\bcapital for my business\b",
#             r"\bexpand my company\b",
#             r"\bexpand my business\b",
#             r"\bcompany expansion\b",
#             r"\bbusiness expansion\b",
#             r"\bfunds? to expand\b",
#             r"\bcapital for expansion\b"
#         ],
#         "car_loan": [
#             r"\bcar loans?\b",
#             r"\bvehicle loans?\b",
#             r"\bauto loans?\b",
#             r"\bbuy (?:a |my )?(?:new |used )?car\b",
#             r"\bfinance (?:a |my )?(?:new |used )?car\b",
#             r"\bused car financing\b",
#             r"\bcar financing\b"
#         ],
#         "home_loan": [
#             r"\bhome loans?\b",
#             r"\bhouse loans?\b",
#             r"\bmortgage loans?\b",
#             r"\b(?:construct|build|buy|purchase|renovate)\b"
#             r".{0,50}\b(?:house|home|flat|property)\b"
#         ],
#         "education_loan": [
#             r"\beducation loans?\b",
#             r"\bstudent loans?\b",
#             r"\bloan for (?:my )?(?:college fees|studies|tuition fees)\b",
#             r"\bstudent loan\b",
#             r"\bmedical college fees\b",
#             r"\bloan for medical college\b",
#             r"\bcollege fees\b",
#             r"\bengineering college fees\b",
#             r"\bmedical studies\b"
#         ]
#     }

#     # Split at common clause boundaries.
#     clauses = re.split(r"\bbut\b|[.!?;]", text)

#     request_pattern = re.compile(
#         r"\b(?:want|need|require|apply for|looking for|"
#         r"would like|get|take)\b"
#     )

#     negation_pattern = re.compile(
#         r"\b(?:don't|do not|doesn't|does not|"
#         r"not|never|no longer)\b"
#     )

#     # First: find an explicitly requested loan type.
#     for clause in clauses:
#         clause = clause.strip()

#         for request in request_pattern.finditer(clause):
#             requested_part = clause[request.start():]

#             for loan_type, patterns in loan_patterns.items():
#                 for pattern in patterns:
#                     match = re.search(pattern, requested_part)

#                     if match:
#                         prefix = requested_part[:match.start()]

#                         # Ignore a loan type negated before its mention.
#                         if negation_pattern.search(prefix):
#                             continue

#                         return loan_type

#     return None

# import re

# def extract_requested_loan_type(text):

#     if not isinstance(text, str) or not text.strip():
#         return None

#     text = re.sub(r"\s+", " ", text.lower().strip())

#     loan_patterns = {
#         "personal_loan": [
#             r"\bpersonal loans?\b",
#             r"\bunsecured loans?\b",
#             r"\bpersonal expenses\b"
#         ],
#         "business_loan": [
#             r"\bbusiness loans?\b",
#             r"\bworking capital\b",
#             r"\bcapital for my business\b",
#             r"\bexpand my company\b",
#             r"\bexpand my business\b",
#             r"\bcompany expansion\b",
#             r"\bbusiness expansion\b",
#             r"\bfunds? to expand\b",
#             r"\bcapital for expansion\b"
#         ],
#         "car_loan": [
#             r"\bcar loans?\b",
#             r"\bvehicle loans?\b",
#             r"\bauto loans?\b",
#             r"\bcar financing\b",
#             r"\bbuy (?:a |my )?(?:new |used )?car\b",
#             r"\bfinance (?:a |my )?(?:new |used )?car\b",
#             r"\bused car financing\b"
#         ],
#         "home_loan": [
#             r"\bhome loans?\b",
#             r"\bmortgage loans?\b",
#             r"\bhouse loans?\b",
#             r"\b(?:construct|build|buy|purchase|renovate)\b",
#             r".{0,50}\b(?:house|home|flat|property)\b",
#             r"\bbuy a house\b",
#             r"\bbuying a house\b",
#             r"\bpurchase a house\b",
#             r"\bbuy a flat\b",
#             r"\bpurchase a flat\b",
#             r"\bbuy a home\b",
#             r"\bpurchase a property\b",
#             r"\bto buy a house\b",
#             r"\bconstruct (?:a|my|our|the) house\b",
#             r"\bconstruction of (?:a|my|our|the) house\b",
#             r"\bbuild (?:a|my|our|the) house\b",
#             r"\bbuilding (?:a|my|our|the) house\b",
#             r"\bto construct (?:a|my|our|the) house\b",
#             r"\bbuy a home\b",
#             r"\bpurchase a house\b",
#             r"\bpurchase a home\b",
#             r"\bbuy a flat\b",
#             r"\bpurchase a flat\b"
#         ],
#         "education_loan": [
#             r"\beducation loans?\b",
#             r"\bstudent loans?\b",
#             r"\bloan for (?:my )?(?:college fees|studies|tuition fees)\b",
#             r"\bstudent loan\b",
#             r"\bmedical college fees\b",
#             r"\bloan for medical college\b",
#             r"\bcollege fees\b",
#             r"\bengineering college fees\b",
#             r"\bmedical studies\b"
#         ]
#     }

#     # Split the message into clauses.
#     clauses = re.split(r"\bbut\b|[.!?;]", text)

#     negation_pattern = re.compile(
#         r"\b(?:don't|do not|doesn't|does not|"
#         r"not|never|no longer)\b"
#         r"\b(?:don't|do not|doesn't|does not|"
#         r"not|never|no longer)\b"
#     )

#     request_pattern = re.compile(
#         r"\b(?:want|need|require|apply for|"
#         r"looking for|would like|prefer|choose)\b"
#         r"\b(?:want|need|require|apply for|looking for|"
#         r"would like|get|take)\b"
#     )

#     # If the user is explicitly choosing between types,
#     # don't guess which one they want.
#     if re.search(
#         r"\b(?:either|or|undecided|not sure)\b",
#         text
#     ):
#         mentioned_types = set()

#         for loan_type, patterns in loan_patterns.items():
#             if any(re.search(p, text) for p in patterns):
#                 mentioned_types.add(loan_type)

#         if len(mentioned_types) > 1:
#             return None

#     # Look for a positive request in each clause.
#     for clause in clauses:
#         clause = clause.strip()

#         if not clause:
#             continue

#         for request in request_pattern.finditer(clause):
#             prefix = clause[:request.start()]
#             requested_part = clause[request.start():]

#             # Skip requests negated immediately beforehand.
#             if negation_pattern.search(prefix):
#                 continue

#             matches = []

#             for loan_type, patterns in loan_patterns.items():
#                 if any(re.search(p, requested_part) for p in patterns):
#                     matches.append(loan_type)

#             # Return only if the request identifies one type.
#             if len(matches) == 1:
#                 return matches[0]

#             if len(matches) > 1:
#                 return None

#     return None

def extract_requested_loan_type(text):

    if not isinstance(text, str) or not text.strip():
        return None

    text = re.sub(r"\s+", " ", text.lower().strip())

    loan_patterns = {
        "personal_loan": [
            r"\bpersonal loans?\b",
            r"\bunsecured loans?\b",
            r"\bpersonal expenses\b"
        ],

        "business_loan": [
            r"\bbusiness loans?\b",
            r"\bworking capital\b",
            r"\bcapital for my business\b",
            r"\bexpand my company\b",
            r"\bexpand my business\b",
            r"\bcompany expansion\b",
            r"\bbusiness expansion\b",
            r"\bfunds? to expand\b",
            r"\bcapital for expansion\b",
            r"\bstart a business\b",
            r"\bstartup funding\b"
        ],

        "car_loan": [
            r"\bcar loans?\b",
            r"\bvehicle loans?\b",
            r"\bauto loans?\b",
            r"\bcar financing\b",
            r"\bcar finance\b",
            r"\bbuy (?:a |my )?(?:new |used )?car\b",
            r"\bfinance (?:a |my )?(?:new |used )?car\b",
            r"\bused car financing\b"
        ],

        "home_loan": [
            r"\bhome loans?\b",
            r"\bmortgage loans?\b",
            r"\bhouse loans?\b",
            r"\b(?:construct|build|buy|purchase|renovate)"
            r"\s+(?:a|my|our|the)\s+"
            r"(?:house|home|flat|property)\b",
            r"\b(?:construction|renovation) of "
            r"(?:a|my|our|the)\s+"
            r"(?:house|home|flat|property)\b",
            r"\b(?:buying|building|constructing|renovating)"
            r"\s+(?:a|my|our|the)\s+"
            r"(?:house|home|flat|property)\b",
            r"\bhouse construction\b",
            r"\bhome construction\b",
            r"\bproperty purchase\b"
        ],

        "education_loan": [
            r"\beducation loans?\b",
            r"\bstudent loans?\b",
            r"\bloan for (?:my )?"
            r"(?:college fees|studies|tuition fees)\b",
            r"\bmedical college fees\b",
            r"\bloan for medical college\b",
            r"\bcollege fees\b",
            r"\bengineering college fees\b",
            r"\bmedical studies\b",
            r"\bstudy abroad\b",
            r"\bhigher studies\b"
        ]
    }

    # Split at contrast words and sentence boundaries.
    clauses = re.split(
        r"\bbut\b|\bhowever\b|[.!?;]",
        text
    )

    request_pattern = re.compile(
        r"\b(?:want|need|require|"
        r"apply for|looking for|would like|"
        r"prefer|choose|wish to|get|take)\b"
    )

    negation_pattern = re.compile(
        r"\b(?:don't|do not|doesn't|does not|"
        r"didn't|did not|cannot|can't|"
        r"never|no longer|not)\b"
    )

    # Avoid guessing when the user explicitly expresses uncertainty.
    if re.search(
        r"\b(?:either|or|undecided|not sure|"
        r"cannot decide|can't decide)\b",
        text
    ):
        mentioned_types = {
            loan_type
            for loan_type, patterns in loan_patterns.items()
            if any(re.search(p, text) for p in patterns)
        }

        if len(mentioned_types) > 1:
            return None

    # Check each clause separately so a negated request does not
    # incorrectly cancel a positive request in another clause.
    for clause in clauses:

        clause = clause.strip()

        if not clause:
            continue

        for request in request_pattern.finditer(clause):

            prefix = clause[:request.start()]
            requested_part = clause[request.start():]

            # Ignore requests negated before the request verb.
            if negation_pattern.search(prefix):
                continue

            # Find the loan types mentioned after this request.
            matches = {
                loan_type
                for loan_type, patterns in loan_patterns.items()
                if any(re.search(p, requested_part) for p in patterns)
            }

            if len(matches) == 1:
                return next(iter(matches))

            if len(matches) > 1:
                return None

        # Handle direct statements without a request verb,
        # such as "A loan to construct my house for 15 years."
        matches = {
            loan_type
            for loan_type, patterns in loan_patterns.items()
            if any(re.search(p, clause) for p in patterns)
        }

        if len(matches) == 1:
            return next(iter(matches))

        if len(matches) > 1:
            return None

    return None




In [755]:

test_messages = [
    "I want a personal loan",
    "I need capital for my business",
    "I want to buy a new car",
    "I need a loan to purchase a house",
    "I need a loan for my college fees",
    "Hello, how are you?",
    "I don't want a car loan",
    "I am not interested in a personal loan",
    "I don't want a car loan, but I need a home loan",
    "I don't want a car loan"

    
]

for message in test_messages:
    print(f"Message: {message}")
    print(f"Loan type: {extract_requested_loan_type(message)}")
    print("-" * 40)

Message: I want a personal loan
Loan type: personal_loan
----------------------------------------
Message: I need capital for my business
Loan type: business_loan
----------------------------------------
Message: I want to buy a new car
Loan type: car_loan
----------------------------------------
Message: I need a loan to purchase a house
Loan type: home_loan
----------------------------------------
Message: I need a loan for my college fees
Loan type: education_loan
----------------------------------------
Message: Hello, how are you?
Loan type: None
----------------------------------------
Message: I don't want a car loan
Loan type: car_loan
----------------------------------------
Message: I am not interested in a personal loan
Loan type: personal_loan
----------------------------------------
Message: I don't want a car loan, but I need a home loan
Loan type: car_loan
----------------------------------------
Message: I don't want a car loan
Loan type: car_loan
----------------------

In [756]:
tenure_units = {
    "year":"years",
    "years":"years",
    "yr":"years",
    "yrs":"years",
    "month":"months",
    "months":"months",
    "mo":"months",
    "mos":"months"
}
print(tenure_units)

{'year': 'years', 'years': 'years', 'yr': 'years', 'yrs': 'years', 'month': 'months', 'months': 'months', 'mo': 'months', 'mos': 'months'}


In [757]:
# def extract_tenure(text):
#     text = text.lower().strip()

#     pattern = (
#         r"\b(\d+)\s*[- ]?\s*"
#         r"(years?|yrs?|months?|mos?)\b"
#     )

#     match = re.search(pattern, text)

#     if not match:
#         return None

#     value = int(match.group(1))
#     unit = match.group(2)

#     # Normalize the unit
#     unit = tenure_units[unit]

#     # Convert tenure into months
#     if unit == "years":
#         tenure_months = value * 12
#     else:
#         tenure_months = value
        
#     if value<=0:
#         return None
    
#     max_tenure_months = 40*12
    
#     if tenure_months > max_tenure_months:
#         return None
    

#     return {
#         "value": value,
#         "unit": unit,
#         "tenure_months": tenure_months
#     }

# def extract_tenure(text):

#     # Step 1: Validate input
#     if not isinstance(text, str) or not text.strip():
#         return None

#     text = re.sub(r"\s+", " ", text.lower().strip())

#     # Step 2: Define the duration pattern
#     duration_pattern = (
#         r"(?P<value>\d+)\s*-?\s*"
#         r"(?P<unit>years?|yrs?|months?|mos?)\b",
#         r"\b(\d+)\s*[- ]?\s*",
#         r"(years?|yrs?|months?|mos?)\b",
#         r"\b(?:over|for|of|within|in|"
#         r"tenure\s*(?:of)?|period\s*(?:of)?)?\s*"
#         r"(\d+)\s*[- ]?\s*"
#         r"(years?|yrs?|months?|mos?)\b"
#     )

#     # Step 3: Search loan-related contexts first
#     patterns = [
#         # Loan tenure is 5 years
#         r"\b(?:loan\s+)?(?:tenure|term|duration)"
#         r"\s*(?:of|is|:)?\s*" + duration_pattern,

#         # Loan for 5 years; loan of ₹5 lakh for 5 years
#         r"\b(?:loan|mortgage)\b.{0,80}?"
#         r"\b(?:for|over)\s*" + duration_pattern,

#         # Repayment period of 36 months
#         r"\brepayment\s+period\s*(?:of|is|:)?\s*"
#         + duration_pattern,

#         # Repay in 36 months; EMI over 5 years
#         r"\b(?:repay|repayment|emi)\b.{0,40}?"
#         r"\b(?:in|over|within|for)\s*" + duration_pattern
#     ]

#     # Step 4: Find a matching duration
#     for pattern in patterns:
#         match = re.search(pattern, text)

#         if not match:
#             continue

#         value = int(match.group("value"))
#         raw_unit = match.group("unit")

#         # Step 5: Validate duration
#         if value <= 0:
#             return None

#         # Step 6: Normalize units
#         if raw_unit.startswith(("year", "yr")):
#             unit = "years"
#             tenure_months = value * 12
#         else:
#             unit = "months"
#             tenure_months = value

#         # Step 7: Apply a broad sanity limit
#         # Actual maximum tenure depends on the lender and loan type.
#         if tenure_months > 40 * 12:
#             return None

#         return {
#             "value": value,
#             "unit": unit,
#             "tenure_months": tenure_months
#         }

#     return None


import re

def extract_tenure(text):

    if not isinstance(text, str) or not text.strip():
        return None

    text = re.sub(r"\s+", " ", text.lower().strip())

    patterns = [
        r"\b(?:over|for|within|in|of)\s+(\d+)\s*[- ]?\s*(years?|yrs?|months?|mos?)\b",
        r"\b(\d+)\s*[- ]?\s*(years?|yrs?|months?|mos?)\s+(?:tenure|period)\b",
        r"\btenure\s+(?:of\s+)?(\d+)\s*[- ]?\s*(years?|yrs?|months?|mos?)\b",
        r"\b(\d+)\s*[- ]?\s*(years?|yrs?|months?|mos?)\b"
    ]

    match = None

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            break

    if not match:
        return None

    value = int(match.group(1))
    raw_unit = match.group(2)

    if value <= 0:
        return None

    if raw_unit.startswith(("year", "yr")):
        unit = "years"
        tenure_months = value * 12
    else:
        unit = "months"
        tenure_months = value

    # Generic upper limit: 40 years.
    if tenure_months > 480:
        return None

    return {
        "value": value,
        "unit": unit,
        "tenure_months": tenure_months
    }




In [758]:
# test_cases = [
#     "I need a loan for 5 years",
#     "I need a loan for 18 months",
#     "I need a loan for 2 yrs",
#     "I need a loan for 0 months",
#     "I need a loan for 500 years",
#     "I need a personal loan"
# ]

# for sentence in test_cases:
#     print(sentence, "->", extract_tenure(sentence))

In [759]:

def extract_interest_rate(text):

    # Step 1: Validate and normalize input
    if not isinstance(text, str) or not text.strip():
        return None

    text = text.lower().strip()

    # Step 2: Define the percentage pattern
    rate_number_pattern = (
        r"(?<![\d.-])"
        r"(?P<rate>\d+(?:\.\d+)?)"
        r"\s*(?:%|percent(?:age)?\b)"
    )

    # Step 3: Define context-aware patterns
    patterns = [
        # Example: interest rate is 9.5%
        re.compile(
            r"\b(?:interest\s+rate|rate\s+of\s+interest|"
            r"loan\s+(?:interest\s+)?rate|"
            r"annual\s+interest(?:\s+rate)?|interest)\b"
            r"[^.!?]{0,40}?"
            + rate_number_pattern
        ),

        # Examples: 10.75 percent interest
        #           9% annual interest
        re.compile(
            rate_number_pattern
            + r"\s*(?:(?:per\s+annum|p\.?\s*a\.?|"
              r"annual(?:ly)?|yearly)\s*)?"
              r"(?:loan\s+)?interest\b"
        ),

        # Example: The rate is 12% per annum
        re.compile(
            r"\brate\b[^.!?]{0,30}?"
            + rate_number_pattern
            + r"[^.!?]{0,15}"
              r"\b(?:per\s+annum|p\.?\s*a\.?|"
              r"annually|annual|yearly|per\s+year)\b"
        )
    ]

    # Step 4: Find a rate mentioned in the correct context
    for pattern in patterns:

        match = pattern.search(text)

        if not match:
            continue

        # Step 5: Extract the numeric rate
        rate = float(match.group("rate"))

        # Step 6: Reject negative rates
        preceding_text = text[:match.start("rate")]

        if re.search(r"-\s*$", preceding_text):
            continue

        # Step 7: Detect whether the rate is annual
        context = text[
            max(0, match.start() - 35):
            min(len(text), match.end() + 35)
        ]

        annual_pattern = (
            r"\b(?:per\s+annum|p\.?\s*a\.?|"
            r"annually|annual|yearly|per\s+year)(?!\w)"
        )

        if re.search(annual_pattern, context):
            period = "annual"
        else:
            period = None

        # Step 8: Return structured information
        return {
            "value": rate,
            "unit": "%",
            "period": period
        }

    return None

In [760]:
test_messages = [
    "My interest rate is 9.5%",
    "The loan has an interest rate of 8%",
    "I was offered 10.75 percent interest",
    "The rate is 12% per annum",
    "My interest rate is 9.5% per annum",
    "What is the interest rate?",
    "I need a loan of ₹5 lakh for 5 years",
    "My CIBIL score is 750",
    "The interest rate is -5%"
]

for message in test_messages:
    print("Input:", message)
    print("Output:", extract_interest_rate(message))
    print("-" * 50)

Input: My interest rate is 9.5%
Output: {'value': 9.5, 'unit': '%', 'period': None}
--------------------------------------------------
Input: The loan has an interest rate of 8%
Output: {'value': 8.0, 'unit': '%', 'period': None}
--------------------------------------------------
Input: I was offered 10.75 percent interest
Output: {'value': 10.75, 'unit': '%', 'period': None}
--------------------------------------------------
Input: The rate is 12% per annum
Output: {'value': 12.0, 'unit': '%', 'period': 'annual'}
--------------------------------------------------
Input: My interest rate is 9.5% per annum
Output: {'value': 9.5, 'unit': '%', 'period': 'annual'}
--------------------------------------------------
Input: What is the interest rate?
Output: None
--------------------------------------------------
Input: I need a loan of ₹5 lakh for 5 years
Output: None
--------------------------------------------------
Input: My CIBIL score is 750
Output: None
--------------------------------

In [761]:
credit_keywors = [
    'cibil score',
    'credit score'
]

In [762]:

import re

credit_keywords = [
    "cibil score",
    "credit score"
]

def extract_credit_score(text):

    # Validate and normalize input
    if not isinstance(text, str) or not text.strip():
        return None

    text = text.lower().strip()

    # Score after the keyword
    pattern_after = (
        r"\b(?:cibil|credit)\s+score\b"
        r"\s*(?:is|of|:)?\s*"
        r"(?P<score>\d{3})\b"
    )

    # Score before the keyword
    pattern_before = (
        r"\b(?P<score>\d{3})\s+"
        r"(?:cibil|credit)\s+score\b"
    )

    match = re.search(pattern_after, text)

    if not match:
        match = re.search(pattern_before, text)

    if not match:
        return None

    score = int(match.group("score"))

    # Validate CIBIL-style score range
    if 300 <= score <= 900:
        return score

    return None


In [763]:
test_messages = [
    "My CIBIL score is 750",
    "My credit score is 720",
    "CIBIL score: 680",
    "I have a 780 CIBIL score",
    "My score is 740 according to CIBIL",
    "I earn ₹60,000 monthly",
    "I want a loan for 5 years",
    "What is a good CIBIL score?",
    "My CIBIL score is 950"
]


for message in test_messages:
    print("Input:",message)
    print("output:",extract_credit_score(message))
    print("-"*50)

Input: My CIBIL score is 750
output: 750
--------------------------------------------------
Input: My credit score is 720
output: 720
--------------------------------------------------
Input: CIBIL score: 680
output: 680
--------------------------------------------------
Input: I have a 780 CIBIL score
output: 780
--------------------------------------------------
Input: My score is 740 according to CIBIL
output: None
--------------------------------------------------
Input: I earn ₹60,000 monthly
output: None
--------------------------------------------------
Input: I want a loan for 5 years
output: None
--------------------------------------------------
Input: What is a good CIBIL score?
output: None
--------------------------------------------------
Input: My CIBIL score is 950
output: None
--------------------------------------------------


In [764]:
dependents_keywords = [
    "dependent",
    "dependents",
    "financially dependent",
    "depend on me"
]

In [765]:
def extract_dependents(text):

    # Step 1: Validate and normalize input
    if not isinstance(text, str) or not text.strip():
        return None

    text = text.lower().strip()

    # Step 2: Define supported numbers
    number_words = {
        "zero": 0,
        "one": 1,
        "two": 2,
        "three": 3,
        "four": 4,
        "five": 5,
        "six": 6,
        "seven": 7,
        "eight": 8,
        "nine": 9,
        "ten": 10,
        "eleven": 11,
        "twelve": 12,
        "thirteen": 13,
        "fourteen": 14,
        "fifteen": 15,
        "sixteen": 16,
        "seventeen": 17,
        "eighteen": 18,
        "nineteen": 19,
        "twenty": 20
    }

    count_token = (
        r"\d+|"
        r"zero|one|two|three|four|five|six|seven|eight|nine|"
        r"ten|eleven|twelve|thirteen|fourteen|fifteen|"
        r"sixteen|seventeen|eighteen|nineteen|twenty"
    )

    # Step 3: Check explicit zero-dependent statements first
    zero_patterns = [
        r"\b(?:i\s+)?(?:do\s+not|don't|dont)\s+"
        r"have\s+(?:any\s+)?(?:financial\s+)?dependents?\b",

        r"\b(?:i\s+)?have\s+"
        r"(?:no|zero)\s+(?:financial\s+)?dependents?\b",

        r"\b(?:no|zero)\s+(?:financial\s+)?dependents?\b",

        r"\bno\s+one\s+(?:is\s+)?"
        r"(?:financially\s+)?dependent\s+on\s+"
        r"(?:me|my\s+income|my\s+salary)\b"
    ]

    for pattern in zero_patterns:
        if re.search(pattern, text):
            return 0

    # Step 4: Define patterns for dependent counts
    patterns = [
        # Examples: 2 dependents; one financial dependent
        r"\b(?P<count>" + count_token + r")\s+"
        r"(?:(?:financial|family)\s+)?dependents?\b",

        # Examples: dependents: 2; number of dependents is three
        r"\b(?:number\s+of\s+)?dependents?\s*"
        r"(?:is|are|:|=)?\s*"
        r"(?P<count>" + count_token + r")\b",

        # Example: 2 people depend on my income
        r"\b(?P<count>" + count_token + r")\s+"
        r"(?:people|persons|family\s+members)\s+"
        r"(?:are\s+)?(?:financially\s+)?dependent\s+on\s+"
        r"(?:me|my\s+income|my\s+salary)\b",

        # Examples: I support 2 people; financially support 3 people
        r"\b(?:financially\s+)?support\s+"
        r"(?P<count>" + count_token + r")\s+"
        r"(?:people|persons|family\s+members)\b",

        # Example: I have 2 children dependent on me
        r"\b(?P<count>" + count_token + r")\s+"
        r"children\s+(?:are\s+)?dependent\s+on\s+"
        r"(?:me|my\s+income|my\s+salary)\b"
    ]

    # Step 5: Extract the count from a matching pattern
    for pattern in patterns:
        match = re.search(pattern, text)

        if not match:
            continue

        raw_count = match.group("count")

        # Step 6: Convert digits or number words to integers
        if raw_count.isdigit():
            count = int(raw_count)
        else:
            count = number_words[raw_count]

        # Step 7: Validate the count
        if count < 0:
            return None

        # Step 8: Return the extracted count
        return count

    # Step 9: No dependent information found
    return None


In [766]:

test_messages = [
    "I have 2 dependents",
    "My number of dependents is three",
    "Dependents: 4",
    "I have one financial dependent",
    "I have no dependents",
    "I don't have any dependents",
    "No one is financially dependent on my income",
    "2 people depend on my income",
    "I financially support 3 people",
    "I have 2 children dependent on me",
    "I want a loan for 5 years",
    "My CIBIL score is 750",
    "I want a loan of ₹5 lakh for 5 years",
    None
]

for message in test_messages:
    print(
        "Input:", message,
        "-> Dependents:", extract_dependents(message)
    )


Input: I have 2 dependents -> Dependents: 2
Input: My number of dependents is three -> Dependents: 3
Input: Dependents: 4 -> Dependents: 4
Input: I have one financial dependent -> Dependents: 1
Input: I have no dependents -> Dependents: 0
Input: I don't have any dependents -> Dependents: 0
Input: No one is financially dependent on my income -> Dependents: 0
Input: 2 people depend on my income -> Dependents: None
Input: I financially support 3 people -> Dependents: 3
Input: I have 2 children dependent on me -> Dependents: 2
Input: I want a loan for 5 years -> Dependents: None
Input: My CIBIL score is 750 -> Dependents: None
Input: I want a loan of ₹5 lakh for 5 years -> Dependents: None
Input: None -> Dependents: None


In [767]:
existing_loan_keywords = [
    "existing loan",
    "existing loans",
    "active loan",
    "active loans",
    "current loan",
    "current loans",
    "already have a loan",
    "already have a personal loan",
    "currently paying",
    "paying emi",
    "loan emi",
    "outstanding loan",
    "outstanding loans",
    "ongoing loan",
    "ongoing loans",
    "have a personal loan",
    "have a home loan",
    "have a car loan",
    "have an education loan",
    "have a business loan",
    "paying a home loan",
    "paying a car loan"
]

no_existing_loan_keywords = [
    "no existing loan",
    "no existing loans",
    "no active loan",
    "no active loans",
    "no current loan",
    "no current loans",
    "don't have any loan",
    "don't have any loans",
    "do not have any loan",
    "do not have any loans",
    "don't have a loan",
    "do not have a loan",
    "no outstanding loan",
    "no outstanding loans",
    "not having any loan",
    "all loans are paid off",
    "loan is fully repaid",
    "loans are fully repaid",
    "previous loan is closed",
    "previous loan was closed",
    "loan has been closed",
    "loan is closed",
    "loan is paid off",
    "loan has been repaid"
]

In [768]:
# def extract_existing_loan(text):

#     if not isinstance(text, str) or not text.strip():
#         return {
#             "status": "unknown",
#             "active_loan_mentioned": None,
#             "closed_loan_mentioned": None
#         }

#     text = text.lower().strip()
#     text = re.sub(r"\s+", " ", text)

#     # Detect clear references to closed or repaid loans
#     closed_patterns = [
#         r"\b(?:loan|loans)\b.{0,40}\b(?:paid off|repaid|closed|settled)\b",
#         r"\b(?:paid off|repaid|closed|settled)\b.{0,40}\b(?:loan|loans)\b",
#         r"\b(?:i|we)\s+(?:have|hold)\s+no\s+"
#         r"(?:active|current|existing|outstanding)?\s*loans?\b",
#         r"\bno active loans?\b",
#         r"\bno current loans?\b",
#         r"\bno existing loans?\b",
#         r"\bno outstanding loans?\b"
# ]

    

#     # Detect clear references to active or outstanding loans
#     active_patterns = [
#         r"\b(?:active|ongoing|outstanding|unpaid)\b.{0,30}\bloan\b",
#         r"\b(?:currently paying|repaying|paying emi)\b.{0,50}\b(?:loan|emi)\b",
#         r"\b(?:have|hold|owe)\b.{0,40}\b(?:an? )?(?:existing|current|active)\b.{0,20}\bloan\b",
#         r"\b(?:have|hold|owe)\b.{0,30}\b(?:personal|home|car|education|business)\s+loan\b"
#     ]

#     closed = any(re.search(p, text) for p in closed_patterns)
#     active = any(re.search(p, text) for p in active_patterns)

#     if active and closed:
#         status = "mixed"
#     elif active:
#         status = "active"
#     elif closed:
#         status = "closed"
#     else:
#         status = "unknown"

#     return {
#         "status": status,
#         "active_loan_mentioned": True if active else None,
#         "closed_loan_mentioned": True if closed else None
#     }


# def extract_existing_loan(text):

#     # Step 1: Validate input
#     if not isinstance(text, str) or not text.strip():
#         return {
#             "status": "unknown",
#             "active_loan_mentioned": None,
#             "closed_loan_mentioned": None
#         }

#     text = re.sub(r"\s+", " ", text.lower().strip())

#     # Step 2: Explicit statements that there is no current loan
#     no_loan_patterns = [
#         r"\bno existing loans?\b",
#         r"\bno active loans?\b",
#         r"\bno current loans?\b",
#         r"\bno outstanding loans?\b",
#         r"\b(?:i|we) don't have (?:any |an? )?"
#         r"(?:existing |active |current )?loans?\b",
#         r"\b(?:i|we) do not have (?:any |an? )?"
#         r"(?:existing |active |current )?loans?\b",
#         r"\bno loans? at all\b"
#     ]

#     # Step 3: Detect closed or repaid loans
#     closed_patterns = [
#         r"\b(?:loan|loans)\b.{0,50}"
#         r"\b(?:paid off|fully repaid|closed|settled)\b",
#         r"\b(?:paid off|fully repaid|closed|settled)\b.{0,50}"
#         r"\b(?:loan|loans)\b",
#         r"\bprevious loan was closed\b",
#         r"\bloan has been repaid\b"
#     ]

#     # Step 4: Detect active loans
#     active_patterns = [
#         r"\b(?:active|ongoing|outstanding|unpaid)\b.{0,30}\bloan\b",
#         r"\b(?:currently paying|paying emi|repaying)\b.{0,50}"
#         r"\b(?:loan|emi)\b",
#         r"\b(?:have|hold|owe)\b.{0,40}"
#         r"\b(?:existing|current|active)\b.{0,20}\bloan\b",
#         r"\b(?:have|hold|owe)\b.{0,30}"
#         r"\b(?:personal|home|car|education|business)\s+loan\b",
#         r"\bmy\s+(?:personal|home|car|education|business)\s+"
#         r"loan\s+is\s+active\b"
#     ]

#     # Step 5: Check no-loan statements separately
#     explicitly_none = any(
#         re.search(pattern, text)
#         for pattern in no_loan_patterns
#     )

#     # Split clauses so a rejected/closed loan in one clause
#     # doesn't automatically cancel an active loan in another.
#     clauses = re.split(r"\bbut\b|[.!?;]", text)

#     active = False
#     closed = False

#     for clause in clauses:
#         clause = clause.strip()

#         if not clause:
#             continue

#         # Detect active references in this clause
#         clause_active = any(
#             re.search(pattern, clause)
#             for pattern in active_patterns
#         )

#         # Ignore active-loan patterns if the clause explicitly
#         # denies having that loan.
#         clause_denies_loan = bool(
#             re.search(
#                 r"\b(?:no|don't have|do not have|"
#                 r"don't owe|do not owe)\b.{0,35}\b"
#                 r"(?:existing |active |current |outstanding |"
#                 r"personal |home |car |education |business )?loans?\b",
#                 clause
#             )
#         )

#         if clause_active and not clause_denies_loan:
#             active = True

#         if any(
#             re.search(pattern, clause)
#             for pattern in closed_patterns
#         ):
#             closed = True

#     # Step 6: Choose status
#     if active and closed:
#         status = "mixed"
#     elif active:
#         status = "active"
#     elif closed:
#         status = "closed"
#     elif explicitly_none:
#         status = "none"
#     else:
#         status = "unknown"

#     # Step 7: Return consistent structure
#     return {
#         "status": status,
#         "active_loan_mentioned": (
#             True if active else False if explicitly_none else None
#         ),
#         "closed_loan_mentioned": (
#             True if closed else None
#         )
#     }


def extract_existing_loan(text):

    unknown_result = {
        "status": "unknown",
        "active_loan_mentioned": None,
        "closed_loan_mentioned": None,
        "outstanding_amount": None,
        "emi_amount": None
    }

    if not isinstance(text, str) or not text.strip():
        return unknown_result

    text = re.sub(r"\s+", " ", text.lower().strip())

    # 1. Detect explicit absence of current loans
    no_loan_patterns = [
        r"\bno existing loans?\b",
        r"\bno active loans?\b",
        r"\bno current loans?\b",
        r"\bno outstanding loans?\b",
        r"\b(?:i|we) don't have (?:any |an? )?"
        r"(?:existing |active |current )?loans?\b",
        r"\b(?:i|we) do not have (?:any |an? )?"
        r"(?:existing |active |current )?loans?\b",
        r"\bno loans? at all\b"
    ]

    explicitly_none = any(
        re.search(p, text) for p in no_loan_patterns
    )

    # 2. Detect closed or repaid loans
    closed_patterns = [
        r"\b(?:loan|loans)\b.{0,50}"
        r"\b(?:paid off|fully repaid|closed|settled)\b",
        r"\b(?:paid off|fully repaid|closed|settled)\b.{0,50}"
        r"\b(?:loan|loans)\b"
    ]

    # 3. Detect active loans
    active_patterns = [
        r"\b(?:active|ongoing|outstanding|unpaid)\b.{0,30}\bloan\b",
        r"\b(?:currently paying|paying emi|repaying)\b.{0,50}"
        r"\b(?:loan|emi)\b",
        r"\b(?:have|hold|owe)\b.{0,40}"
        r"\b(?:existing|current|active)\b.{0,20}\bloan\b",
        r"\b(?:have|hold|owe)\b.{0,30}"
        r"\b(?:personal|home|car|education|business)\s+loan\b",
        r"\bmy\s+(?:personal|home|car|education|business)\s+"
        r"loan\s+is\s+active\b"
    ]

    clauses = re.split(r"\bbut\b|[.!?;]", text)
    active = False
    closed = False

    for clause in clauses:
        clause = clause.strip()

        if not clause:
            continue

        denies_loan = bool(re.search(
            r"\b(?:no|don't have|do not have|"
            r"don't owe|do not owe)\b.{0,35}\b"
            r"(?:existing |active |current |outstanding |"
            r"personal |home |car |education |business )?loans?\b",
            clause
        ))

        if not denies_loan and any(
            re.search(p, clause) for p in active_patterns
        ):
            active = True

        if any(re.search(p, clause) for p in closed_patterns):
            closed = True

    # 4. Determine status
    if active and closed:
        status = "mixed"
    elif active:
        status = "active"
    elif closed:
        status = "closed"
    elif explicitly_none:
        status = "none"
    else:
        status = "unknown"

    # 5. Define money patterns for existing loan amounts
    money_pattern = (
        r"(?P<currency>₹|rs\.?|inr|\$|usd)?\s*"
        r"(?P<number>\d+(?:,\d{3})*(?:\.\d+)?)\s*"
        r"(?P<unit>lakhs?|lacs?|crores?|k|thousands?|millions?)?"
    )

    outstanding_patterns = [
        r"\b(?:outstanding balance|outstanding amount|"
        r"remaining loan amount|remaining balance|"
        r"loan balance|balance remaining|"
        r"outstanding loan)\b"
        r"\s*(?:is|of|:|=|about|around)?\s*"
        + money_pattern
    ]

    emi_patterns = [
        r"\b(?:monthly emi|emi amount|loan emi|"
        r"emi|monthly instalment|monthly installment)\b"
        r"\s*(?:is|of|:|=|about|around)?\s*"
        + money_pattern
    ]

    def find_amount(patterns):
        for pattern in patterns:
            match = re.search(pattern, text)

            if match:
                amount = normalize_money_value(
                    match.groupdict()
                )

                if amount is not None and amount > 0:
                    return amount

        return None

    outstanding_amount = find_amount(outstanding_patterns)
    emi_amount = find_amount(emi_patterns)

    # 6. Return status and amounts separately
    return {
        "status": status,
        "active_loan_mentioned": (
            True if active else False if explicitly_none else None
        ),
        "closed_loan_mentioned": True if closed else None,
        "outstanding_amount": outstanding_amount,
        "emi_amount": emi_amount
    }


In [769]:
test_messages = [
    "I have an existing personal loan",
    "I am currently paying a home loan",
    "I already have a car loan",
    "I have no existing loans",
    "I don't have any active loans",
    "I have paid off my previous loan",
    "I want to apply for a new loan",
    "I don't want a personal loan",
    "I have no personal loan, but I have a car loan",
    "I am paying EMI for my education loan",
    "My home loan is active",
    "I have a loan, but it is fully paid off",
    "I have two loans and one is closed",
    "Tell me about personal loans",
    ""
]

for message in test_messages:
    print("Input:", repr(message))
    print("Output:", extract_existing_loan(message))
    print("-" * 50)

Input: 'I have an existing personal loan'
Output: {'status': 'active', 'active_loan_mentioned': True, 'closed_loan_mentioned': None, 'outstanding_amount': None, 'emi_amount': None}
--------------------------------------------------
Input: 'I am currently paying a home loan'
Output: {'status': 'active', 'active_loan_mentioned': True, 'closed_loan_mentioned': None, 'outstanding_amount': None, 'emi_amount': None}
--------------------------------------------------
Input: 'I already have a car loan'
Output: {'status': 'active', 'active_loan_mentioned': True, 'closed_loan_mentioned': None, 'outstanding_amount': None, 'emi_amount': None}
--------------------------------------------------
Input: 'I have no existing loans'
Output: {'status': 'none', 'active_loan_mentioned': False, 'closed_loan_mentioned': None, 'outstanding_amount': None, 'emi_amount': None}
--------------------------------------------------
Input: "I don't have any active loans"
Output: {'status': 'none', 'active_loan_mentione

In [770]:
# def extract_entities(text):
#     applicant_data = {
#         "income": extract_income(text),
#         "income_frequency": detect_income_frequency(text),
#         "loan_amount": extract_loan_amount(text),
#         "loan_type": extract_requested_loan_type(text),
#         "tenure": extract_tenure(text),
#         "interest_rate": extract_interest_rate(text),
#         "credit_score": extract_credit_score(text),
#         "dependents": extract_dependents(text),
#         "existing_loan": extract_existing_loan(text)
#     }
    
#     return applicant_data


def extract_entities(text):

    # Validate input
    if not isinstance(text, str) or not text.strip():
        return {
            "income": None,
            "income_frequency": None,
            "loan_amount": None,
            "loan_type": None,
            "tenure": None,
            "interest_rate": None,
            "credit_score": None,
            "dependents": None,
            "existing_loan": {
                "status": "unknown",
                "active_loan_mentioned": None,
                "closed_loan_mentioned": None
            }
        }

    # Extract income and frequency together
    income_result = extract_income(text)

    if isinstance(income_result, dict):
        income = income_result.get("income")
        income_frequency = income_result.get("frequency")
    else:
        income = income_result
        income_frequency = detect_income_frequency(text)

    # Extract remaining entities
    applicant_data = {
        "income": income,
        "income_frequency": income_frequency,
        "loan_amount": extract_loan_amount(text),
        "loan_type": extract_requested_loan_type(text),
        "tenure": extract_tenure(text),
        "interest_rate": extract_interest_rate(text),
        "credit_score": extract_credit_score(text),
        "dependents": extract_dependents(text),
        "existing_loan": extract_existing_loan(text)
    }

    return applicant_data


In [771]:

test_messages = [
    "I don't want a personal loan. I need a car loan of ₹6 lakh for 5 years.",

    "I need either a personal loan or a business loan of ₹10 lakh for 5 years.",

    "I have an existing home loan, but I want a personal loan of ₹3 lakh for 3 years.",

    "My old personal loan is closed, and now I want a new car loan of ₹7 lakh for 5 years.",

    "I earn ₹60,000 per month and need ₹5 lakh. I haven't decided which type of loan I want.",

    "My salary is ₹50,000 per month, and I want a personal loan. I don't know my CIBIL score and haven't decided the tenure."
]


for i, message in enumerate(test_messages, start=1):
    print(f"\nTEST CASE {i}")
    print("Message:", repr(message))

    result = extract_entities(message)

    for entity, value in result.items():
        print(f"{entity}: {value}")

    print("-" * 60)



TEST CASE 1
Message: "I don't want a personal loan. I need a car loan of ₹6 lakh for 5 years."
income: None
income_frequency: None
loan_amount: 600000.0
loan_type: personal_loan
tenure: {'value': 5, 'unit': 'years', 'tenure_months': 60}
interest_rate: None
credit_score: None
dependents: None
existing_loan: {'status': 'unknown', 'active_loan_mentioned': None, 'closed_loan_mentioned': None, 'outstanding_amount': None, 'emi_amount': None}
------------------------------------------------------------

TEST CASE 2
Message: 'I need either a personal loan or a business loan of ₹10 lakh for 5 years.'
income: None
income_frequency: None
loan_amount: 1000000.0
loan_type: None
tenure: {'value': 5, 'unit': 'years', 'tenure_months': 60}
interest_rate: None
credit_score: None
dependents: None
existing_loan: {'status': 'unknown', 'active_loan_mentioned': None, 'closed_loan_mentioned': None, 'outstanding_amount': None, 'emi_amount': None}
------------------------------------------------------------

T

In [772]:
import re

text = "I earn ₹1.5 lakh monthly and need ₹25 lakh to expand my company over 7 years."

pattern = (
    r"\b(?:need|want|require)\s+"
    r"(?P<currency>₹|rs\.?|inr|\$|usd)?\s*"
    r"(?P<number>\d+(?:,\d{3})*(?:\.\d+)?)\s*"
    r"(?P<unit>lakhs?|lacs?|crores?|k|thousand|millions?)?"
)

match = re.search(pattern, text, re.IGNORECASE)

if match:
    print(match.groupdict())
else:
    print("No match")

{'currency': '₹', 'number': '25', 'unit': 'lakh'}
